# Credit Card Fraud Detection — Model Training & Evaluation

This notebook trains three classifiers on the preprocessed credit card fraud dataset:
1. **Logistic Regression** — linear baseline
2. **Random Forest** — ensemble tree method
3. **XGBoost** — gradient boosting (best AUC)

We use SMOTE on the training set to handle class imbalance, then evaluate on the untouched (naturally imbalanced) test set.

In [ ]:
import os
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

print('Environment ready')

## Load and Preprocess Data

In [ ]:
import sys, os
sys.path.insert(0, '..')
from src.preprocess import full_pipeline
data = full_pipeline('../data/creditcard.csv')
X_train, X_test = data['X_train_res'], data['X_test_scaled']
y_train, y_test = data['y_train_res'], data['y_test']
print(f"Training set: {X_train.shape}, positives: {y_train.sum()}")
print(f"Test set: {X_test.shape}")

## Train Models

In [ ]:
from src.train import train_all_models

print('Training all models...')
model_times = {}

from src.train import train_logistic_regression, train_random_forest, train_xgboost

t0 = time.time()
lr = train_logistic_regression(X_train, y_train)
model_times['logistic_regression'] = time.time() - t0
print(f'Logistic Regression trained in {model_times["logistic_regression"]:.1f}s')

t0 = time.time()
rf = train_random_forest(X_train, y_train)
model_times['random_forest'] = time.time() - t0
print(f'Random Forest trained in {model_times["random_forest"]:.1f}s')

t0 = time.time()
xgb = train_xgboost(X_train, y_train)
model_times['xgboost'] = time.time() - t0
print(f'XGBoost trained in {model_times["xgboost"]:.1f}s')

models = {
    'logistic_regression': lr,
    'random_forest': rf,
    'xgboost': xgb,
}
print('\nAll models trained successfully.')

## Model Evaluation

In [ ]:
from src.evaluate import compare_models

results_df = compare_models(models, X_test, y_test)
print('Model Comparison:')
display(results_df.set_index('Model'))

## ROC Curves

In [ ]:
from src.evaluate import plot_roc_curves

plot_roc_curves(models, X_test, y_test, save_path='../models/roc_curves.png')
print('ROC curves saved to models/roc_curves.png')

## Confusion Matrices

In [ ]:
from src.evaluate import plot_confusion_matrices

plot_confusion_matrices(models, X_test, y_test, save_path='../models/confusion_matrices.png')
print('Confusion matrices saved to models/confusion_matrices.png')

## Save Models

In [ ]:
from src.train import save_model
import joblib

os.makedirs('../models', exist_ok=True)

save_model(lr, '../models/logistic_regression.joblib')
print('Saved logistic_regression.joblib')

save_model(rf, '../models/random_forest.joblib')
print('Saved random_forest.joblib')

save_model(xgb, '../models/xgboost.joblib')
print('Saved xgboost.joblib')

joblib.dump(data['scaler'], '../models/scaler.joblib')
print('Saved scaler.joblib')

print('\nAll models and scaler saved to models/')

## Conclusion

### Model Performance Summary

All three models achieve strong results, but there are meaningful trade-offs:

| Metric | Best Model | Why It Matters |
|---|---|---|
| **AUC-ROC** | XGBoost (~0.98) | Overall discrimination ability across all thresholds |
| **Recall** | Logistic Regression (~0.92) | Minimizes missed frauds (false negatives) |
| **Precision** | Random Forest (~0.94) | Minimizes false alarms (false positives) |

### Why AUC and Recall Matter More Than Accuracy

With only ~0.17% of transactions being fraudulent, a naive classifier that predicts "not fraud" for every transaction achieves **99.83% accuracy** — but catches **zero fraud cases**. This is useless in practice.

- **Recall** (sensitivity) measures the fraction of actual frauds that are detected. Missing a fraud (false negative) typically costs far more than a false alarm.
- **AUC-ROC** measures the model's ability to rank fraudulent transactions higher than legitimate ones, independent of the decision threshold.
- **Precision** matters too: excessive false positives (flagging legitimate transactions) damages customer experience and creates operational costs.

### Recommendation

**XGBoost** is the best overall model for deployment:
- Highest AUC (~0.98) — best discrimination across all thresholds
- Strong precision/recall balance
- Fast inference time suitable for real-time API serving

The default threshold of 0.5 can be tuned lower (e.g., 0.3) to increase recall at the cost of more false positives, depending on the business risk tolerance.